In [2]:
import os
import glob
import pytesseract
from PIL import Image
from jiwer import cer, wer


# ============================================================
# PATHS
# ============================================================

# YOUR WORD DATASET TEST FOLDER
data_dir = r"\\wsl.localhost\Ubuntu\home\janmejaywsl\akshar_training\synthetic_word_dataset"
# data_dir = r"\\wsl$\Ubuntu\home\janmejaywsl\tesstrain\data\guj-ground-truth"

# ------------------------------------------------------------
# BASELINE MODEL
# ------------------------------------------------------------
baseline_model_dir = (
    r"D:\code material 2.0\akshar-setu-v2\model\baseline"
)


# ------------------------------------------------------------
# WORD-DATASET FINE-TUNED MODEL
# ------------------------------------------------------------
word_dataset_model_dir = (
    r"D:\code material 2.0\akshar-setu-v2\model\word\synthetic"
)


# ============================================================
# STORAGE
# ============================================================

ground_truths = []

baseline_predictions = []
word_dataset_predictions = []


# ============================================================
# GET WORD IMAGES
# ============================================================

print("Gathering word dataset images...")

image_files = glob.glob(
    os.path.join(data_dir, "*.png")
)

# ------------------------------------------------------------
# Evaluate first 200 images
# Change to:
# image_files = image_files
# if you want the complete dataset.
# ------------------------------------------------------------

image_files = image_files[:200]

print(
    f"Starting word-dataset evaluation "
    f"on {len(image_files)} images...\n"
)


# ============================================================
# EVALUATION
# ============================================================

for idx, img_path in enumerate(image_files):

    # --------------------------------------------------------
    # Ground-truth file
    #
    # Example:
    # word_001.png
    # word_001.gt.txt
    # --------------------------------------------------------

    gt_path = img_path.replace(
        ".png",
        ".gt.txt"
    )

    if not os.path.exists(gt_path):

        print(
            f"WARNING: Ground truth missing for "
            f"{os.path.basename(img_path)}"
        )

        continue


    # --------------------------------------------------------
    # READ GROUND TRUTH
    # --------------------------------------------------------

    with open(
        gt_path,
        "r",
        encoding="utf-8"
    ) as f:

        true_text = f.read().strip()


    # --------------------------------------------------------
    # LOAD IMAGE
    # --------------------------------------------------------

    img = Image.open(img_path)


    # ========================================================
    # STORE GROUND TRUTH
    # ========================================================

    ground_truths.append(true_text)


    # ========================================================
    # MODEL 1 — BASELINE
    # ========================================================

    os.environ["TESSDATA_PREFIX"] = baseline_model_dir

    baseline_text = pytesseract.image_to_string(
        img,
        lang="guj"
    ).strip()

    baseline_predictions.append(
        baseline_text
    )


    # ========================================================
    # MODEL 2 — WORD DATASET MODEL
    # ========================================================

    os.environ["TESSDATA_PREFIX"] = word_dataset_model_dir

    word_dataset_text = pytesseract.image_to_string(
        img,
        lang="guj"
    ).strip()

    word_dataset_predictions.append(
        word_dataset_text
    )


    # ========================================================
    # PROGRESS
    # ========================================================

    if (idx + 1) % 20 == 0:

        print(
            f"Processed "
            f"{idx + 1}/{len(image_files)} images..."
        )


# ============================================================
# METRIC FUNCTION
# ============================================================

def calculate_metrics(
    ground_truth,
    predictions
):

    # --------------------------------------------------------
    # Character Error Rate
    # --------------------------------------------------------

    character_error_rate = cer(
        ground_truth,
        predictions
    )


    # --------------------------------------------------------
    # Word Error Rate
    # --------------------------------------------------------

    word_error_rate = wer(
        ground_truth,
        predictions
    )


    # --------------------------------------------------------
    # Character Accuracy
    # --------------------------------------------------------

    character_accuracy = (
        1 - character_error_rate
    )


    # --------------------------------------------------------
    # Word Accuracy
    # --------------------------------------------------------

    word_accuracy = (
        1 - word_error_rate
    )


    # --------------------------------------------------------
    # Exact Match Accuracy
    # --------------------------------------------------------

    exact_matches = sum(
        gt.strip() == pred.strip()
        for gt, pred in zip(
            ground_truth,
            predictions
        )
    )

    exact_match_accuracy = (
        exact_matches / len(ground_truth)
        if len(ground_truth) > 0
        else 0
    )


    return {
        "CER": character_error_rate,
        "WER": word_error_rate,
        "Character Accuracy": character_accuracy,
        "Word Accuracy": word_accuracy,
        "Exact Match Accuracy": exact_match_accuracy
    }


# ============================================================
# BASELINE RESULTS
# ============================================================

baseline_results = calculate_metrics(
    ground_truths,
    baseline_predictions
)


# ============================================================
# WORD DATASET RESULTS
# ============================================================

word_dataset_results = calculate_metrics(
    ground_truths,
    word_dataset_predictions
)


# ============================================================
# IMPROVEMENT
# ============================================================

cer_improvement = (
    (
        baseline_results["CER"]
        - word_dataset_results["CER"]
    )
    / baseline_results["CER"]
    * 100
    if baseline_results["CER"] > 0
    else 0
)


wer_improvement = (
    (
        baseline_results["WER"]
        - word_dataset_results["WER"]
    )
    / baseline_results["WER"]
    * 100
    if baseline_results["WER"] > 0
    else 0
)


# ============================================================
# FINAL ACCURACY MATRIX
# ============================================================

print("\n")

print("=" * 85)

print(
    "               GUJARATI WORD DATASET OCR EVALUATION"
)

print("=" * 85)

print(
    f"Images evaluated: {len(ground_truths)}"
)

print()

print(
    f"{'Metric':<28}"
    f"{'BASELINE':>20}"
    f"{'WORD-DATASET MODEL':>30}"
)

print("-" * 85)


# ------------------------------------------------------------
# CER
# ------------------------------------------------------------

print(
    f"{'CER':<28}"
    f"{baseline_results['CER'] * 100:>19.2f}%"
    f"{word_dataset_results['CER'] * 100:>29.2f}%"
)


# ------------------------------------------------------------
# WER
# ------------------------------------------------------------

print(
    f"{'WER':<28}"
    f"{baseline_results['WER'] * 100:>19.2f}%"
    f"{word_dataset_results['WER'] * 100:>29.2f}%"
)


# ------------------------------------------------------------
# Character Accuracy
# ------------------------------------------------------------

print(
    f"{'Character Accuracy':<28}"
    f"{baseline_results['Character Accuracy'] * 100:>19.2f}%"
    f"{word_dataset_results['Character Accuracy'] * 100:>29.2f}%"
)


# ------------------------------------------------------------
# Word Accuracy
# ------------------------------------------------------------

print(
    f"{'Word Accuracy':<28}"
    f"{baseline_results['Word Accuracy'] * 100:>19.2f}%"
    f"{word_dataset_results['Word Accuracy'] * 100:>29.2f}%"
)


# ------------------------------------------------------------
# Exact Match Accuracy
# ------------------------------------------------------------

print(
    f"{'Exact Match Accuracy':<28}"
    f"{baseline_results['Exact Match Accuracy'] * 100:>19.2f}%"
    f"{word_dataset_results['Exact Match Accuracy'] * 100:>29.2f}%"
)


print("=" * 85)


# ============================================================
# IMPROVEMENT SUMMARY
# ============================================================

print()

print("=" * 85)

print(
    "                    IMPROVEMENT SUMMARY"
)

print("=" * 85)


print(
    f"Baseline CER             : "
    f"{baseline_results['CER'] * 100:.2f}%"
)

print(
    f"Word Dataset CER         : "
    f"{word_dataset_results['CER'] * 100:.2f}%"
)

print(
    f"CER Improvement          : "
    f"{cer_improvement:.2f}%"
)

print()


print(
    f"Baseline WER             : "
    f"{baseline_results['WER'] * 100:.2f}%"
)

print(
    f"Word Dataset WER         : "
    f"{word_dataset_results['WER'] * 100:.2f}%"
)

print(
    f"WER Improvement          : "
    f"{wer_improvement:.2f}%"
)

print("=" * 85)

Gathering word dataset images...
Starting word-dataset evaluation on 200 images...

Processed 20/200 images...
Processed 40/200 images...
Processed 60/200 images...
Processed 80/200 images...
Processed 100/200 images...
Processed 120/200 images...
Processed 140/200 images...
Processed 160/200 images...
Processed 180/200 images...
Processed 200/200 images...


               GUJARATI WORD DATASET OCR EVALUATION
Images evaluated: 200

Metric                                  BASELINE            WORD-DATASET MODEL
-------------------------------------------------------------------------------------
CER                                        4.79%                        11.43%
WER                                       13.50%                        35.00%
Character Accuracy                        95.21%                        88.57%
Word Accuracy                             86.50%                        65.00%
Exact Match Accuracy                      86.50%                        65.00%

  

test image


In [5]:
import os
import pytesseract
from PIL import Image

os.environ['TESSDATA_PREFIX'] = r"D:\code material 2.0\akshar-setu-v2\model\word\synthetic"

image_path1 = r"D:\code material 2.0\akshar-setu\data\raw\line1.png"
image_path2 = r"D:\code material 2.0\akshar-setu\data\raw\line2.png"
image_path3 = r"D:\code material 2.0\akshar-setu\data\raw\para1.jpeg"

try:
    img = Image.open(image_path3)
    text = pytesseract.image_to_string(img, lang='guj', config='--psm 3').strip()

    print("--- OLD MODEL PREDICTION ---")
    print(text if text else "[No text detected]")

except Exception as e:
    print(f"Error loading image or running Tesseract: {e}")

--- OLD MODEL PREDICTION ---
.

રનકો સનાુષ માે મઝવશર્ ાનદ બનુબવતો તો. ગમી જ ગેવ, સાં ઉકપડધરા કરેી
તબરી કોઈ તમા કતી ડવ કસમષતનામાતનતનેગલો વો ડતો કતુંનું૫
ઈબુંપત. પરત મતાબે ુનનેો ાવવાવી નેેચર નો મક ેવાીઈ
અનેતારપછીબે વરષ વાીતીગયાં છતાંતેનીમાતાપાછીઆાવી નહી.

ા કં ગઈજ બે ૩ક કમુનાદયમાં સતત ?યા ૩તો તો.

ેઈકતંગતેનામાઈકઈકડતુડમનેેઈકઈકેઃ"તે
ા કવાગઈછનો કડતોે:ગતોમરાગઈ

પ મનેલીધાવગર એ કેમ ગઈ સુમાુધનીએઆસુબરી કરયાદસોનીઆંમાંાંસુલાવતી
તા કસુધીગેફોદ કકાયકલબેઈછટેશદો ૧પપઈી તોમાવશેજ ે

ગપસાંનીસો કોઈતેની સમે જોઈ ૨રેં. કવતગાંખઉપર ૃડંકતુંગને કેઈ વત
તે કકેજવબગાપતુંઃ

ડ, વશેા નો, ો

ેટલો વબબાકનંગેંગરમા શૂિ ેતો. તે થેતો, તો, સતો ચાર-ાચદિવશે૧વળીપાોે
ગેનો ુછતો છેવટેશે ેપવનુંપન ોં કનુ. મસપનો રવમનુખો ૩વર
કા તને ની માિટો પાોડોયએવીમાતા તેનાદદયમાંજનમીમને ે એકલોએકલો ?વાાો
મઉમાંતેકેઈવર વાઉકતોામા"

તેનો ૫તા૩૭બકાે ૧ગીઉકતો મનેેનેશશરે ય ફેવતો

ેકક ેે ુદદ મુમવાનીકોઈીમાં ?ઈ. મનું કેઈ ૫પશ દાવી ુતીમાંજવાતે
તો માનાસું ૧ંપડંવાસનેતેધરષરીનેેતો વીકોઈી મનવાઆવાઢે
માં રડ મગરકતો માનીનનો ૧ો ાબકમામેન સમજ મેમાની ોેષમોબ
કતો ડતો.

બ સીઓો વાજતાર

In [ ]:
%%sql
